# 03 Quality Assurance & Final Validation
This notebook evaluates the statistical quality of the synthetic data and performs final business rule validations.

In [ ]:
import pandas as pd
import numpy as np
import os
from sdmetrics.reports.single_table import DiagnosticReport, QualityReport

# Load data
df_prod = pd.read_csv('../data/out/csv/dim_products.csv')
df_fact = pd.read_csv('../data/out/csv/fact_invoices_flat.csv')

print(f"Loaded {len(df_prod)} products and {len(df_fact)} transaction rows.")

## 1. Rule R2: Inventory Iterative Back-filling
Ensure stock_on_hand >= total_sold + buffer.

In [ ]:
def perform_backfilling(df_p, df_f, buffer=50):
    # Calculate total sold per product
    total_sold = df_f.groupby('Mã hàng')['Số lượng'].sum().reset_index()
    total_sold.columns = ['Mã hàng', 'Total_Sold']
    
    # Merge with products
    df_merged = df_p.merge(total_sold, on='Mã hàng', how='left').fillna(0)
    
    # Update stock_on_hand if current stock < total_sold + buffer
    mask = df_merged['Tồn kho'] < (df_merged['Total_Sold'] + buffer)
    df_merged.loc[mask, 'Tồn kho'] = df_merged.loc[mask, 'Total_Sold'] + buffer + np.random.randint(10, 100, size=mask.sum())
    
    # Drop the helper column
    df_updated = df_merged.drop(columns=['Total_Sold'])
    return df_updated

df_prod_updated = perform_backfilling(df_prod, df_fact)

# Guard: Ensure all product codes in FACT exist in DIM
fact_codes = df_fact['Mã hàng'].unique()
prod_codes = df_prod_updated['Mã hàng'].unique()
missing = set(fact_codes) - set(prod_codes)

if not missing:
    print("Inventory Check: PASS (All transaction products exist in DIM)")
    # Overwrite snapshots with consistent data
    df_prod_updated.to_csv('../data/out/csv/dim_products.csv', index=False)
    df_prod_updated.to_excel('../data/out/excel/DIM_PRODUCTS_Import_Consistent.xlsx', index=False)
else:
    print(f"Inventory Check: FAIL (Missing codes: {missing})")

## 2. Statistical Quality Evaluation (Diagnostic)
Using sdmetrics to check for data structure and logical consistency.

In [ ]:
# Placeholder for actual Real vs Synthetic comparison
# In a full pipeline, we would load the 'real' seed data here
print("Ready to run SDMetrics QualityReport on generated fact data.")

## 3. Final Import Readiness Checklist

In [ ]:
def final_sanity_check(df_f):
    errors = []
    if df_f['Mã hóa đơn'].isnull().any(): errors.append("Missing Invoice Codes")
    if not df_f['Mã hóa đơn'].str.startswith('HDIP').all(): errors.append("Rule R1 Violation: Prefix HDIP missing")
    if (df_f['Số lượng'] <= 0).any(): errors.append("Rule R3 Violation: Non-positive quantity")
    
    if not errors:
        print("Final Sanity Check: ALL PASS. Ready for KiotViet Import.")
    else:
        print(f"Final Sanity Check: FAILED. Errors: {errors}")

final_sanity_check(df_fact)